In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import tensorly as tl
tl.set_backend('numpy')

In [3]:
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from hoda.classification import ZScore, BTTDACV, SelectFdrMin1
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
import tensorly as tl
import os

cv=StratifiedKFold(random_state=42, shuffle=True)
pipelines=dict()

clf = make_pipeline(
    FunctionTransformer(tl.to_numpy),
    StandardScaler(),
    SelectFdrMin1(alpha=0.05),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

bttda_params=dict(
        hoda_params=dict(
            max_iter=128,
            toeplitz=None,
            taper=False,
            verbose=True,
            refit_shrinkage=False,
        ),
        clf=clf, verbose=True,
        cv=cv,
        #n_jobs=5*11
        n_jobs = 1
)

pipelines['HODA'] = Pipeline([
    ('tensorly', FunctionTransformer(tl.tensor)),
    ('zscore1', ZScore()),
    ('bttda',BTTDACV(
        max_n_blocks=1,
        thetas=[0,0.1,0.2,0.3,0.4,0.5,0.6, 0.7, 0.8, 0.9,1],
        **bttda_params)
    ),
    ('clf', clf)
])

pipelines['PARAFACDA'] = Pipeline([
    ('tensorly', FunctionTransformer(tl.tensor)),
    ('zscore1', ZScore()),
    ('bttda',BTTDACV(
        max_n_blocks=16,
        thetas=[0],
        **bttda_params)
    ),
    ('clf', clf)
])

pipelines['BTTDA'] = Pipeline([
    ('tensorly', FunctionTransformer(tl.tensor)),
    ('zscore1', ZScore()),
    ('bttda',BTTDACV(
        max_n_blocks=16,
        thetas=[0,0.1,0.2,0.3,0.4,0.5,0.6, 0.7, 0.8, 0.9,1],
        **bttda_params)
    ),
    ('clf', clf)
])


In [4]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

datasets = [
BNCI2014_008(),
#BNCI2014_009(),
#BNCI2015_003(),
#BI2012(),
#BI2013a(),
#BI2014a(),
#BI2014b(),
#BI2015a(),
#BI2015b(),
#Cattan2019_VR(),
#ErpCore2021_ERN(),
#ErpCore2021_LRP(),
#ErpCore2021_MMN(),
#ErpCore2021_N170(),
#ErpCore2021_N400(),
#ErpCore2021_P3(),
#Lee2019_ERP(),
]

paradigm = P300(
    resample=48,

)
cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)

In [5]:
import copy
from sklearn.base import clone
import dask
import os
from sklearn.preprocessing import FunctionTransformer
import tensorly as tl

def eval_moabb_within_session(dataset, subject, pipe):
    subj_dataset = copy.deepcopy(dataset)
    n_subjects = len(dataset.subject_list)
    subj_dataset.subject_list = [subject]
    evaluation = WithinSessionEvaluation(
        paradigm=paradigm,
        datasets=subj_dataset,
        overwrite=False,
        random_state=42,
        #n_jobs=5,
        n_jobs=1,
        suffix=f'bttda_dask_{dataset.code}_{subject}_{pipe}',
        cache_config=cache_config,
    )
    print(f'dataset={dataset.code}, subject={subject}/{n_subjects}, pipe={pipe}')
    return evaluation.process({pipe:clone(pipelines[pipe])})



# Build the task graph
job_args = []
for dataset in datasets:
    for subject in dataset.subject_list:
        for pipe in pipelines.keys():
            job_args.append((dataset, subject,pipe))
    


In [6]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import pickle
from distributed.protocol.serialize import register_serialization_family
import os
from dask.distributed import LocalCluster

timeout = 12*60*60

def create_cluster(cluster='slurm', scale=256*2):
    env = {
        'TENSORLY_BACKEND': 'numpy',
        'TENSORLY_TENALG_BACKEND': 'einsum',
        #'CUPY_ACCELERATORS': 'cutensor,cub',
    }
    job_script_prologue=[]
    tl.initialize_backend()
    for k,v in env.items():
        os.environ[k] = v
        job_script_prologue += [f'export {key}="{value}"' for key, value in env.items()]        
    

    
    if cluster=='local':
        cluster = LocalCluster(
            n_workers=1,
            threads_per_worker=1,
        )       
    elif cluster=='debug':
        cluster = SLURMCluster(
            cores=1,
            processes=1,
            memory="4GB",
            account='llonpp',
            queue='gpu_p100_debug',
            walltime='00:15:00',
            scheduler_options=dict(
                dashboard_address=':8787'
            ),
            job_extra_directives=[
                '-M genius',
                '-o logs/slurm-%A.log',
                '--gpus-per-node=1',
                '--nodes=1',
                '--export=ALL',
        
            ],
            local_directory=os.path.join(os.environ['VSC_SCRATCH'],'.cache'),
            death_timeout=timeout,
            job_script_prologue=job_script_prologue
            #interface='ib1',

        )
        cluster.scale(1)
    elif cluster=='gpu':
        cluster = SLURMCluster(
            cores=1,
            processes=1,
            memory="40GB",
            account='llonpp',
            queue='gpu_p100',
            walltime='02:00:00',
            scheduler_options=dict(
                dashboard_address=':8787'
            ),
            job_extra_directives=[
                '-M genius',
                '-o logs/slurm-%A.log',
                '--gpus-per-node=1',
                '--nodes=1',
                '--export=ALL',
            ],
            local_directory=os.path.join(os.environ['VSC_SCRATCH'],'.cache'),
            death_timeout=timeout,
            job_script_prologue=job_script_prologue
            #interface='ib1',
        )
        cluster.scale(scale)
    elif cluster=='cpu':
        cluster = SLURMCluster(
            cores=96,
            memory="250GB",
            account='llonpp',
            queue='batch_sapphirerapids',
            walltime='02:00:00',
            scheduler_options=dict(
                dashboard_address=':8787'
            ),
            job_extra_directives=[
                '-M wice',
                '-o logs/slurm-%A.log',
                '--nodes=1',
                '--export=ALL',
            ],
            local_directory=os.path.join(os.environ['VSC_SCRATCH'],'.cache'),
            death_timeout=timeout,
            job_script_prologue=job_script_prologue
        )
        cluster.scale(scale)
    else:
        raise ValueError
        
    return cluster


def create_client(cluster):
    return Client(cluster)

In [ ]:
import joblib
from joblib import Parallel, delayed
import distributed
from IPython import display
import pandas as pd


with create_cluster(cluster='local') as cluster, create_client(cluster) as client:
    cluster.scale(8)
    #with joblib.parallel_backend('dask', wait_for_workers_timeout=timeout): 
    #    results = Parallel(n_jobs=-1, verbose=True)(delayed(eval_moabb_within_session)(*args) for args in job_args)
    for args in job_args:
        eval_moabb_within_session(*args)
results = pd.concat(results, ignore_index=True)

dataset=BNCI2014-008, subject=1/8, pipe=HODA


BNCI2014-008-WithinSession:   0%|          | 0/1 [00:00<?, ?it/s]

No hdf5_path provided, models will not be saved.
fold=0, theta=0
Fitting block 1/1...


/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/util.py:31: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  w,v = scipy.sparse.linalg.eigsh(
Backward HODA model rank=(1, 1):  17%|█▋        | 22/128 [00:00<00:00, 193.17it/s]
/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/util.py:31: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  w,v = scipy.sparse.linalg.eigsh(


fold=0, theta=0, n_blocks=1
fold=0, theta=0.1
Fitting block 1/1...


...

Forward model :  11%|█         | 14/127 [00:00<00:00, 366.97it/s]


Fitting block 3/16...


Forward model :   9%|▉         | 12/127 [00:00<00:00, 371.95it/s]


Fitting block 4/16...


Forward model :  13%|█▎        | 16/127 [00:00<00:00, 391.60it/s]

In [ ]:
results.to_csv('results/moabb_erp.csv')
results

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean')